# Exploratory Data Analysis
## Indonesia E-Commerce Sales

**Final Project & Portfolio — Topik 7**

**Nama:** Maulana Aldi Pradana  
**Kelas:** XII TJKT C  
**Project:** Sales Performance Analysis

Notebook ini mendokumentasikan proses Exploratory Data Analysis (EDA)
menggunakan Final Clean Dataset hasil tahap Data Preparation.

### Tujuan
- Memahami karakteristik transaction value dan volume transaksi.
- Menganalisis tren penjualan dari waktu ke waktu.
- Menganalisis performa kategori produk dan provinsi.
- Menganalisis metode pembayaran.
- Menganalisis cancellation dan return.
- Mengidentifikasi pola quantity dan transaction value.
- Menghasilkan insight yang dapat digunakan untuk Business Analysis.


## 1. Import Library & Load Dataset

Dataset clean menggunakan separator `;`.

Raw dataset tidak digunakan untuk modifikasi pada tahap ini.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(
    "/content/FINAL_PROJECT_MAULANA_ALDI_PRADANA"
)

CLEAN_FILE = (
    BASE_DIR
    / "02_Data"
    / "Clean_Dataset"
    / "Final_Clean_Dataset.csv"
)

df = pd.read_csv(
    CLEAN_FILE,
    sep=";",
    encoding="utf-8-sig"
)

df["order_datetime"] = pd.to_datetime(
    df["order_datetime"],
    errors="coerce"
)

df["Total Pembayaran"] = pd.to_numeric(
    df["Total Pembayaran"],
    errors="coerce"
)

df["Total Diskon"] = pd.to_numeric(
    df["Total Diskon"],
    errors="coerce"
)

df["total_qty"] = pd.to_numeric(
    df["total_qty"],
    errors="coerce"
)

df["total_returned_qty"] = pd.to_numeric(
    df["total_returned_qty"],
    errors="coerce"
)

print("Rows   :", len(df))
print("Columns:", len(df.columns))
print("Duplicate order_id:", df["order_id"].duplicated().sum())


## 2. Dataset Overview

Tahap ini digunakan untuk memeriksa struktur dataset sebelum analisis.


In [ ]:
print("Shape:", df.shape)

print("\nColumns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nDate range:")
print("Min:", df["order_datetime"].min())
print("Max:", df["order_datetime"].max())


## 3. Data Quality Check

Pemeriksaan difokuskan pada duplicate, missing value, dan validitas datetime.


In [ ]:
quality = pd.DataFrame({
    "Missing": df.isna().sum(),
    "Missing_%": (
        df.isna().sum() / len(df) * 100
    )
})

print(quality[
    quality["Missing"] > 0
])

print("\nDuplicate rows:", df.duplicated().sum())
print(
    "Duplicate order_id:",
    df["order_id"].duplicated().sum()
)

print(
    "\nInvalid datetime:",
    df["order_datetime"].isna().sum()
)


## 4. KPI Baseline


In [ ]:
total_orders = df["order_id"].nunique()

total_revenue = (
    df["Total Pembayaran"].sum()
)

total_quantity = (
    df["total_qty"].sum()
)

total_discount = (
    df["Total Diskon"].sum()
)

total_returned_qty = (
    df["total_returned_qty"].sum()
)

cancelled_orders = int(
    df["Is_Cancelled"].sum()
)

returned_orders = int(
    df["Is_Returned"].sum()
)

aov = total_revenue / total_orders

cancellation_rate = (
    cancelled_orders
    / total_orders
    * 100
)

return_rate = (
    returned_orders
    / total_orders
    * 100
)

print(f"Total Revenue       : Rp{total_revenue:,.0f}")
print(f"Total Orders        : {total_orders:,}")
print(f"Total Quantity      : {total_quantity:,.0f}")
print(f"Total Discount      : Rp{total_discount:,.0f}")
print(f"Total Returned Qty  : {total_returned_qty:,.0f}")
print(f"AOV                 : Rp{aov:,.2f}")
print(f"Cancelled Orders    : {cancelled_orders:,}")
print(f"Cancellation Rate   : {cancellation_rate:.2f}%")
print(f"Returned Orders     : {returned_orders:,}")
print(f"Return Rate         : {return_rate:.2f}%")


## 5. Monthly Sales Performance

Analisis digunakan untuk melihat perubahan Revenue, Orders, Quantity,
dan AOV dari waktu ke waktu.


In [ ]:
monthly = (
    df.assign(
        Periode=df["order_datetime"].dt.to_period("M")
    )
    .groupby("Periode")
    .agg(
        Revenue=("Total Pembayaran", "sum"),
        Orders=("order_id", "nunique"),
        Quantity=("total_qty", "sum")
    )
    .reset_index()
)

monthly["AOV"] = (
    monthly["Revenue"]
    / monthly["Orders"]
)

monthly["Revenue_Growth_%"] = (
    monthly["Revenue"].pct_change() * 100
)

monthly["Order_Growth_%"] = (
    monthly["Orders"].pct_change() * 100
)

display(monthly)


## 6. Revenue, Orders, and AOV Relationship

Korelasi digunakan untuk melihat hubungan statistik antar-metrik.
Korelasi tidak digunakan sebagai bukti hubungan sebab-akibat.


In [ ]:
corr_revenue_orders = (
    monthly["Revenue"]
    .corr(monthly["Orders"])
)

corr_revenue_aov = (
    monthly["Revenue"]
    .corr(monthly["AOV"])
)

print(
    f"Revenue vs Orders correlation: "
    f"{corr_revenue_orders:.4f}"
)

print(
    f"Revenue vs AOV correlation: "
    f"{corr_revenue_aov:.4f}"
)

print("\nHighest Revenue:")
display(
    monthly.sort_values(
        "Revenue",
        ascending=False
    ).head(5)
)


## 7. Product Category Performance


In [ ]:
category = (
    df.groupby(
        "product_categories"
    )
    .agg(
        Revenue=(
            "Total Pembayaran",
            "sum"
        ),
        Orders=(
            "order_id",
            "nunique"
        ),
        Quantity=(
            "total_qty",
            "sum"
        ),
        Returned_Quantity=(
            "total_returned_qty",
            "sum"
        )
    )
    .reset_index()
)

category["AOV"] = (
    category["Revenue"]
    / category["Orders"]
)

category["Revenue_Share_%"] = (
    category["Revenue"]
    / total_revenue
    * 100
)

top10_category = (
    category
    .sort_values(
        "Revenue",
        ascending=False
    )
    .head(10)
)

display(top10_category)


In [ ]:
top10_category_revenue = (
    top10_category["Revenue"].sum()
)

top10_category_share = (
    top10_category_revenue
    / total_revenue
    * 100
)

print(
    f"Top 10 category revenue: "
    f"Rp{top10_category_revenue:,.0f}"
)

print(
    f"Top 10 category revenue share: "
    f"{top10_category_share:.2f}%"
)

print("\nHighest revenue category:")
print(
    top10_category.iloc[0][
        [
            "product_categories",
            "Revenue",
            "Orders",
            "Quantity"
        ]
    ]
)


## 8. Regional / Province Performance


In [ ]:
province = (
    df.groupby("Provinsi")
    .agg(
        Revenue=(
            "Total Pembayaran",
            "sum"
        ),
        Orders=(
            "order_id",
            "nunique"
        ),
        Quantity=(
            "total_qty",
            "sum"
        )
    )
    .reset_index()
)

province["AOV"] = (
    province["Revenue"]
    / province["Orders"]
)

province["Revenue_Share_%"] = (
    province["Revenue"]
    / total_revenue
    * 100
)

top10_province = (
    province
    .sort_values(
        "Revenue",
        ascending=False
    )
    .head(10)
)

display(top10_province)


## 9. Payment Method Performance


In [ ]:
payment = (
    df.groupby(
        "Metode_Pembayaran_Clean"
    )
    .agg(
        Revenue=(
            "Total Pembayaran",
            "sum"
        ),
        Orders=(
            "order_id",
            "nunique"
        )
    )
    .reset_index()
)

payment["AOV"] = (
    payment["Revenue"]
    / payment["Orders"]
)

payment["Revenue_Share_%"] = (
    payment["Revenue"]
    / total_revenue
    * 100
)

payment["Order_Share_%"] = (
    payment["Orders"]
    / total_orders
    * 100
)

display(
    payment.sort_values(
        "Revenue",
        ascending=False
    )
)


## 10. Cancellation Analysis


In [ ]:
cancel_summary = pd.DataFrame({
    "Metric": [
        "Total Orders",
        "Cancelled Orders",
        "Cancellation Rate"
    ],
    "Value": [
        total_orders,
        cancelled_orders,
        f"{cancellation_rate:.2f}%"
    ]
})

display(cancel_summary)

print("\nCancellation by reason:")

cancel_reason = (
    df[df["Is_Cancelled"] == True]
    .groupby(
        "Alasan Pembatalan"
    )
    .size()
    .reset_index(
        name="Orders"
    )
    .sort_values(
        "Orders",
        ascending=False
    )
)

cancel_reason["Share_%"] = (
    cancel_reason["Orders"]
    / cancelled_orders
    * 100
)

display(
    cancel_reason.head(10)
)


## 11. Return Analysis


In [ ]:
return_summary = pd.DataFrame({
    "Metric": [
        "Returned Orders",
        "Return Rate",
        "Returned Quantity"
    ],
    "Value": [
        returned_orders,
        f"{return_rate:.2f}%",
        total_returned_qty
    ]
})

display(return_summary)

return_category = (
    df.groupby(
        "product_categories"
    )
    .agg(
        Returned_Quantity=(
            "total_returned_qty",
            "sum"
        ),
        Quantity=(
            "total_qty",
            "sum"
        )
    )
    .reset_index()
    .sort_values(
        "Returned_Quantity",
        ascending=False
    )
)

display(
    return_category.head(10)
)


## 12. Quantity & Order Value Analysis


In [ ]:
quantity_available = df[
    df["total_qty"].notna()
].copy()

quantity_bucket = pd.cut(
    quantity_available["total_qty"],
    bins=[
        0,
        1,
        2,
        5,
        10,
        20,
        float("inf")
    ],
    labels=[
        "1",
        "2",
        "3–5",
        "6–10",
        "11–20",
        ">20"
    ]
)

quantity_available[
    "Quantity_Bucket"
] = quantity_bucket

qty_analysis = (
    quantity_available
    .groupby(
        "Quantity_Bucket",
        observed=False
    )
    .agg(
        Orders=(
            "order_id",
            "nunique"
        ),
        Revenue=(
            "Total Pembayaran",
            "sum"
        ),
        Quantity=(
            "total_qty",
            "sum"
        )
    )
    .reset_index()
)

qty_analysis["AOV"] = (
    qty_analysis["Revenue"]
    / qty_analysis["Orders"]
)

display(qty_analysis)


In [ ]:
corr_revenue_quantity = (
    monthly["Revenue"]
    .corr(monthly["Quantity"])
)

corr_aov_quantity = (
    monthly["AOV"]
    .corr(monthly["Quantity"])
)

print(
    f"Revenue vs Quantity correlation: "
    f"{corr_revenue_quantity:.4f}"
)

print(
    f"AOV vs Quantity correlation: "
    f"{corr_aov_quantity:.4f}"
)


## 13. Key Findings

### Finding 1 — Revenue dan volume transaksi
Revenue memiliki korelasi yang kuat dengan jumlah order dan quantity
pada tingkat bulanan.

### Finding 2 — Konsentrasi revenue
Revenue terkonsentrasi pada sejumlah kategori produk dan provinsi
tertentu.

### Finding 3 — Operational pattern
Cancellation dan return merupakan pola operasional yang perlu
dianalisis lebih lanjut pada tahap Business Analysis.

> Korelasi dan pola EDA bersifat deskriptif. Temuan tidak digunakan
> sebagai bukti hubungan sebab-akibat.


## 14. Data Quality & Limitations

- `total_qty` memiliki 283 missing values dan tidak diimputasi.
- Payment pada data asli memiliki missing values dan dinormalisasi
  menjadi `Tidak Diketahui`.
- `Waktu Pesanan Dibuat` digunakan sebagai datetime utama.
- Terdapat bulan tanpa data pada rentang kalender dataset.
- Dataset tidak menyediakan `Customer_ID`.
- Dataset tidak menyediakan `Cost` dan `Profit`.
- Unique customer, retention, CLV, profit, dan profit margin
  tidak dihitung.
- Kelompok dengan sample kecil perlu diinterpretasikan secara hati-hati.


## 15. Conclusion

EDA memberikan gambaran mengenai pola transaction value, order,
quantity, kategori produk, provinsi, payment method, cancellation,
dan return.

Hasil EDA menjadi dasar untuk tahap berikutnya yaitu:

**Business Analysis → Root Cause Analysis → Recommendation Matrix**

File pendukung EDA disimpan pada folder:

`04_Analysis/`
